# SSE Random Scroll — htmx v4

Migrated from `sse_rand_scroll.py` (htmx v2).

## Migration summary

| Change | htmx v2 | htmx v4 |
|--------|---------|---------|
| **SSE extension** | Separate script: `htmx-ext-sse@2.2.1/sse.js` | Built into core htmx — no extra script |
| **Extension attribute** | `hx_ext="sse"` | Not needed |
| **Connect to stream** | `sse_connect="/number-stream"` | `hx_get="/number-stream"` + `hx_trigger="load"` |
| **Swap event** | `sse_swap="message"` | Not needed — htmx auto-detects `text/event-stream` |
| **Server message format** | `sse_message(data)` (includes `event: message\n`) | `sse_message(data, htmx4=True)` (omits `event:` line) |
| **App init** | `fast_app(hdrs=hdrs)` | `fast_app(htmx=False, htmx4=True)` |

In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import *
from hx4_patch.core import *

In [ ]:
import random
from asyncio import sleep

app,rt = fast_app(htmx=False, htmx4=True)

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate pairs of random numbers, as the list grows scroll downwards."),
        Div(hx_get="/number-stream",
            hx_trigger="load",
            hx_swap="beforeend show:bottom"))

shutdown_event = signal_shutdown()
async def number_generator():
    while not shutdown_event.is_set():
        data = Div(
                Article(random.randint(1, 100)),
                Article(random.randint(1, 100)))
        yield sse_message(data, htmx4=True)
        await sleep(1)

@rt("/number-stream")
async def get(): return EventStream(number_generator())
srv = JupyUvi(app)